In [ ]:
import time, logging
from pathlib import Path
from buddy.experiment import Experiment
from buddy.image import Image
from napari_buddy._reader import read_smfish_json
from glob import glob
import shutil
import numpy as np
import napari 
from skimage import io
from scipy.ndimage import shift
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

***

### 1. Read in microscope images

In [ ]:
exp = Experiment.from_cfg_file(r"config_files\220601\config220601-yET563-BR1.json") #### important file contains threshold values! 
######################################################################################## and other fun parameters!!!!!!!!!!!!!!!!!
exp.inputdir = r"..\..\test yourself\small example set\YET572 BR4"
exp.outputdir = r"..\..\test yourself\small example set\YET572 BR4\output"
exp.create_image_list()

In [ ]:
for params in list(exp.images.values()):
    logging.info(params)
    my_image = Image.from_dict(params, exp)
    print(my_image.experiment.parameters)

    # ~ 01-configure: read raw data and write the input layers that the
    # later stages read back from disk. No focus/spots/masks yet.
    tic = time.time()
    my_image.read_image()            # -> self.mrna (fluorescence channels + DAPI)
    my_image.read_cells()            # -> self.cells (the DIC image)
    my_image.DIC = my_image.cells    # create_grgb expects self.DIC

    my_image.create_grgb()           # -> self.grgb (DIC + DAPI) used for segmentation
    my_image.save_input_layers()     # writes {channel}.tif and DIC.tif
    my_image.time['01-configure'] = time.time() - tic
    my_image.save_metadata("configure")

***

### 2. Segment images

In [ ]:
exp.read_image_list_from_jsons()
exp.init_omnipose()

for f in exp.json_files:
    logging.info(f'image: {f}')
    my_image = Image.from_json(f, exp)

    # rebuild the GRGB composite that segment_cells() needs (from_json only
    # restores metadata, not the image arrays)
    my_image.load_data('mrna')                                  # -> self.mrna[ch]['data']
    my_image.DIC = io.imread(Path(my_image.savepath) / 'DIC.tif')
    my_image.create_grgb()                                      # -> self.grgb

    # segment image (~ 02-segment)
    tic = time.time()
    my_image.segment_cells()
    my_image.time['02-segment-cells'] = time.time() - tic

    tic = time.time()
    my_image.segment_dapi()
    my_image.time['02-segment-dapi'] = time.time() - tic

    # postprocess masks -> DIC_masks_pp.tif (the non-aligned cell masks)
    tic = time.time()
    my_image.postprocess_masks()
    my_image.time['02-segment-pp'] = time.time() - tic

    # indicate in metadata which stage was run last
    my_image.save_metadata("segment")


***

### 3. Detect spots

In [ ]:
exp.read_image_list_from_jsons()

for f in exp.json_files:
    logging.info(f'image: {f}')
    my_image = Image.from_json(f, exp)

    # load the NON-ALIGNED segmented masks + fluorescence channels.
    # find_focus() picks its high-density patch from self.cell_masks_uncorrected
    # (DIC_masks_pp.tif), so these must be in memory before it runs.
    my_image.load_data('masks')      # -> cell_masks_uncorrected (+ corrected/None)
    my_image.load_data('mrna')       # -> mrna[ch]['data']

    # detect spots (~ 03-detect-spots) -- focus is computed on the non-aligned masks
    tic = time.time()
    my_image.find_focus()
    my_image.filter()
    my_image.detect_spots()
    my_image.time['03-detect-spots'] = time.time() - tic

    # decompose spots (~ 04-decompose-spots)
    tic = time.time()
    my_image.decompose_spots()
    my_image.time['04-decompose-spots'] = time.time() - tic

    # align DIC + cell masks to the detected spots. With
    # overwrite_corrected=True this updates corrected/DIC_corrected_masks.tif
    # (backing up the original as *_preAlign.tif) so assign_spots below uses
    # the aligned masks.
    tic = time.time()
    my_image.align(overwrite_corrected=True)
    my_image.time['align'] = time.time() - tic

    # copy segmented DAPI into corrected/ if no corrected version exists yet
    savepath  = f.parent
    corrected = savepath / 'corrected'
    corrected.mkdir(parents=True, exist_ok=True)
    dst = corrected / 'DAPI_corrected_masks.tif'
    src = savepath / 'DAPI_masks.tif'          # segmented (un-corrected) DAPI
    if not dst.is_file():
        if src.is_file():
            shutil.copy2(src, dst)
            print(f'copied DAPI masks -> {dst}')
        else:
            print(f'WARNING: no DAPI_masks.tif to copy for {savepath.name}')

    # assign spots (05-assign-spots) -- reads the now-aligned corrected masks
    my_image.assign_spots()

    # save image (json pickle)
    tic = time.time()
    my_image.save("05")
    my_image.time['05-save'] = time.time() - tic

    my_image.save_metadata("spots")

***

### 4. Correct cell masks and add bud mask

In [ ]:
#select image to correct
i = 0
my_image = Image.from_dict(list(exp.images.values())[i], exp)

# Load every layer from napari_buddy's reader (parses img.json -> napari
# LayerData tuples: DIC, DIC/DAPI masks, each channel, detected + decomposed spots).
#
# max_project=True  -> max-project the z-stacks (DAPI + fluorescent channels) to
#                      2D, make them visible + additive so DIC/DAPI/channels blend,
#                      autoscale each to its own min/max, and flatten the spots to
#                      (y, x) so they overlay the projection.
# max_project=False -> load the full z-stacks unchanged.
max_project = True

def _prep(layer_data, project):
    data, kwargs, ltype = layer_data
    kwargs = dict(kwargs)

    if ltype == 'points':
        # 'label' only exists after assign_spots; fall back to a solid colour
        feats = kwargs.get('features')
        if kwargs.get('border_color') == 'label' and (
                feats is None or 'label' not in getattr(feats, 'columns', [])):
            kwargs['border_color'] = 'red'
        if project and data.shape[1] == 3:
            data = data[:, 1:]                       # drop z -> (y, x)

    elif ltype == 'image' and project:
        if data.ndim == 3:
            data = data.max(axis=0)                  # max-project along z
        kwargs['visible'] = True
        kwargs['blending'] = 'additive'
        lo, hi = float(data.min()), float(data.max())
        if hi > lo:
            kwargs['contrast_limits'] = [lo, hi]     # autoscale

    return data, kwargs, ltype

viewer = napari.current_viewer() or napari.Viewer()
for layer_data in read_smfish_json(str(Path(my_image.savepath) / 'img.json')):
    viewer._add_layer_from_data(*_prep(layer_data, max_project))


In [ ]:
# --- add a bud-annotation layer (background / mother / bud) + suggestions ---
# Seeds a paintable 'bud masks' layer (1 = bud, 2 = background, 3 = mother) from
# the cell segmentation in the 'DIC masks' layer (which the reader loads as the
# corrected mask when present), plus two review overlays:
#   'potential buds'     (yellow rectangles) - small cells, pre-seeded as bud
#   'potentially budded'  (cyan ellipses)    - non-round cells (a mother+bud blob)
# Paint to re-classify: select label 1 = bud, 2 = background, 3 = mother.
# (logic lives in bud_tools.py - place it next to this notebook, or in the buddy
#  package and change the import to `from buddy.bud_tools import ...`)
from bud_tools import classify_cells, build_bud_seed, candidate_rectangles, BUD

cell_masks = np.asarray(viewer.layers['DIC masks'].data)
info = classify_cells(cell_masks, bud_area_fraction=0.5,
                      min_solidity=0.93, max_round_eccentricity=0.85)
print(f"{len(info['bud_labels'])} candidate bud(s) (area < {info['bud_threshold']:.0f} px); "
      f"{len(info['budded_labels'])} non-round / potentially budded cell(s); "
      f"median cell area {info['median_area']:.0f} px.")
print("labels: 1 = bud, 2 = background, 3 = mother")

# preserve buds already annotated (so a bud stays a bud even after its DIC
# cell was merged into a mother)
existing_buds = viewer.layers['bud masks'].data if 'bud masks' in viewer.layers else None
bud_seed = build_bud_seed(cell_masks, info['bud_labels'], keep_bud_mask=existing_buds)

for name in ('bud masks', 'potential buds', 'potentially budded'):
    if name in viewer.layers:
        viewer.layers.remove(viewer.layers[name])

bud_layer = viewer.add_labels(bud_seed, name='bud masks', opacity=0.5)
bud_layer.selected_label = BUD
bud_layer.mode = 'paint'

bud_rects = candidate_rectangles(info['props'], info['bud_labels'])
if bud_rects:
    viewer.add_shapes(bud_rects, name='potential buds', shape_type='rectangle',
                      edge_color='yellow', edge_width=6, face_color='transparent')
budded_rects = candidate_rectangles(info['props'], info['budded_labels'])
if budded_rects:
    viewer.add_shapes(budded_rects, name='potentially budded', shape_type='ellipse',
                      edge_color='cyan', edge_width=6, face_color='transparent')


In [ ]:
# --- link each painted bud to a mother, merge into the DIC mask, box the pairs ---
# Each bud (bud masks == 1) is treated as its own DIC cell and linked to the cell
# it touches most (its mother). The bud is merged into that mother in the DIC mask
# (they end up sharing one label), each mother-bud pair is boxed in magenta, and
# buds with no mother are boxed red. The merged DIC mask overwrites
# corrected/DIC_corrected_masks.tif.
from bud_tools import (assign_buds_to_mothers, merge_buds_into_mothers,
                       mother_bud_pair_rectangles)

dic_masks = np.asarray(viewer.layers['DIC masks'].data)
bud_masks = np.asarray(viewer.layers['bud masks'].data)

bud_labeled, assignments, unassigned, bud_cc = assign_buds_to_mothers(dic_masks, bud_masks)
dic_merged = merge_buds_into_mothers(dic_masks, assignments, bud_cc)
pair_rects, orphan_rects = mother_bud_pair_rectangles(dic_merged, assignments, bud_cc)

n = len(assignments)
print(f"{n} bud(s): {n - len(unassigned)} linked to a mother, {len(unassigned)} unassigned.")
for comp, (mother, own) in assignments.items():
    if mother is not None:
        print(f"  bud {comp} (cell {own}) -> mother cell {mother}")
    else:
        print(f"  bud {comp}: NOT linked to a mother cell")

# apply the merge to the DIC masks layer + overwrite the corrected file
viewer.layers['DIC masks'].data = dic_merged
corrected = Path(my_image.savepath) / 'corrected'
corrected.mkdir(parents=True, exist_ok=True)
io.imsave(corrected / 'DIC_corrected_masks.tif', np.asarray(dic_merged))
print(f"overwrote {corrected / 'DIC_corrected_masks.tif'} (buds merged into mothers)")

for name in ('mother-bud pairs', 'unassigned buds'):
    if name in viewer.layers:
        viewer.layers.remove(viewer.layers[name])
if pair_rects:
    viewer.add_shapes(pair_rects, name='mother-bud pairs', shape_type='rectangle',
                      edge_color='magenta', edge_width=6, face_color='transparent')
if orphan_rects:
    viewer.add_shapes(orphan_rects, name='unassigned buds', shape_type='rectangle',
                      edge_color='red', edge_width=6, face_color='transparent')


In [ ]:
# Save the corrected masks into the corrected/ folder.
# These exact names + location are what assign_spots (section 5) and the reader read:
#   corrected/DIC_corrected_masks.tif, DAPI_corrected_masks.tif, bud_corrected_masks.tif
corrected = Path(my_image.savepath) / 'corrected'
corrected.mkdir(parents=True, exist_ok=True)

io.imsave(corrected / 'DIC_corrected_masks.tif',  np.asarray(viewer.layers['DIC masks'].data))
io.imsave(corrected / 'DAPI_corrected_masks.tif', np.asarray(viewer.layers['DAPI masks'].data))
io.imsave(corrected / 'bud_corrected_masks.tif',  np.asarray(viewer.layers['bud masks'].data))

***

### 5. Reassign spots

In [ ]:
exp.read_image_list_from_jsons()
for f in exp.json_files:
    my_image = Image.from_json(f, exp)
    my_image.load_data('mrna')    # channel data incl. DAPI z-stack (for the nuclear z-projection)
    my_image.load_data('spots')   # detected spots
    my_image.load_data('dense')   # decomposed dense regions
    my_image.assign_spots()       # reads corrected/ masks, rewrites {ch}.csv
    my_image.save_metadata("spots")